# 14. 다른 프로그램과 대화하기 — HTTP와 API 호출

| 구성 | 의미 | LLM API 호출에서의 예 |
| --- | --- | --- |
| **메서드** | 무엇을 하려는가 | `POST` (대화를 보내고 답을 받는다) |
| **주소(URL)** | 어디에 요청하는가 | `https://api.openai.com/v1/chat/completions` |
| **헤더(header)** | 요청에 대한 부가 정보 | 인증 키(`Authorization`), 본문 형식(`Content-Type`) |
| **본문(body)** | 보내는 데이터 | 모델 이름과 메시지 목록 (JSON) |

In [7]:
import requests

url = "https://jsonplaceholder.typicode.com/posts/1"
response = requests.get(url, timeout=10)

print(response.status_code)
print(response.headers["Content-Type"])

data = response.json()
print(type(data))
print(data["id"], data["userId"])
print(data["title"])

200
application/json; charset=utf-8
<class 'dict'>
1 1
sunt aut facere repellat provident occaecati excepturi optio reprehenderit


In [8]:
data

{'userId': 1,
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

In [9]:
data["body"]

'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'

In [10]:
response.headers

{'Date': 'Fri, 27 Nov 2026 03:14:43 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'access-control-allow-credentials': 'true', 'Cache-Control': 'max-age=43200', 'etag': 'W/"124-yiKdLzqO5gfBrJFrcdJ8Yq0LGnU"', 'expires': '-1', 'pragma': 'no-cache', 'vary': 'Origin, Accept-Encoding', 'x-content-type-options': 'nosniff', 'x-powered-by': 'Express', 'x-ratelimit-limit': '3600000', 'x-ratelimit-remaining': '3599992', 'x-ratelimit-reset': '1790812826', 'Age': '10429', 'Nel': '{"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}', 'cf-cache-status': 'HIT', 'Report-To': '{"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=fEMyKwhhPH4ZlnR7YX7uWQQNiN28IftLUIvH1lIxDXPoLhEVgqqDICs42p4%2BfUQkiNtVG0yp1NZVY0dnHH%2FnKhWTeFpj%2BBj0VLfC9poPFlUeWqpxg2HYJm51nlcj9wBpKANvOOBbtdAE1bN3Ol5A"}]}', 'Content-Encoding': 'gzip', 'Server': 'cloudflare', 'CF-RAY': 'a60df6dce8a6061e-LAX', 'alt-svc

In [12]:
response = requests.get(
    "https://jsonplaceholder.typicode.com/comments",
    params={"postId": 1},
    timeout=10
)
print(response.url)

comments = response.json()
print(type(comments), len(comments))
print(comments[0]["email"])

https://jsonplaceholder.typicode.com/comments?postId=1
<class 'list'> 5
Eliseo@gardner.biz


In [16]:
new_post = {
    "title": "연차 신청 질문",
    "body": "연차는 며칠 전까지 신청해야 하나요?",
    "userId": 1,
}
response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=new_post,
    timeout=10,
)
print(response.status_code)
print(response.json())

201
{'title': '연차 신청 질문', 'body': '연차는 며칠 전까지 신청해야 하나요?', 'userId': 1, 'id': 101}


In [18]:
# 예시
import os
import requests

response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}"},
    json={
        "model": "gpt-6-luna",
        "messages": [
            {"role": "system", "content": "너는 사내 규정 도우미다."},
            {"role": "user", "content": "연차는 며칠이야?"},
        ],
    },
    timeout=30,
)
reply = response.json()["choices"][0]["message"]["content"]

KeyError: 'choices'

In [20]:
import requests


def fetch_post(post_id: int) -> dict | None:
    url = f"https://jsonplaceholder.typicode.com/posts/{post_id}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()             # 4xx, 5xx이면 예외를 일으킨다
        return response.json()
    except requests.HTTPError as e:
        print(f"HTTP 에러: {e}")
    except requests.Timeout:
        print("응답 시간 초과")
    except requests.ConnectionError:
        print("서버에 연결할 수 없다")
    return None


print(fetch_post(1)["id"])
print(fetch_post(999))

1
HTTP 에러: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/999
None


In [24]:
try:
    requests.get("http://127.0.0.1:9999", timeout=3)
except requests.ConnectionError as e:
    print("연결 실패:", type(e).__name__)
    print(f'type(e): {e}')
    print(f'e: {e}')
    print(e == type(e))

연결 실패: ConnectionError
type(e): HTTPConnectionPool(host='127.0.0.1', port=9999): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=9999): Failed to establish a new connection: [WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다"))
e: HTTPConnectionPool(host='127.0.0.1', port=9999): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=9999): Failed to establish a new connection: [WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다"))
False


**기본** — `https://jsonplaceholder.typicode.com/users/1`을 요청해 사용자의 이름(`name`)과 회사 이름(`company`의 `name`)을 출력한다.

**확장** — `https://jsonplaceholder.typicode.com/todos`에 `params={"userId": 1}`을 주어 요청하고, 완료된(`completed`가 `True`) 할 일의 개수를 컴프리헨션으로 센다.

**도전** — 12장의 `@retry` 데코레이터를 `requests.ConnectionError`와 `requests.Timeout`에 반응하도록 바꾸고, `fetch_post()`에 붙인다. `127.0.0.1:9999`로 주소를 바꿔 재시도가 일어나는지 확인한다.

In [26]:
user = requests.get("https://jsonplaceholder.typicode.com/users/1", timeout=10).json()
print(user["name"], "/", user["company"]["name"])

todos = requests.get(
    "https://jsonplaceholder.typicode.com/todos",
    params={"userId": 1},
    timeout=10,
).json()
done = [t for t in todos if t["completed"]]
print(len(todos), len(done))

Leanne Graham / Romaguera-Crona
20 11


In [30]:
import functools
import time

def retry(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for attemp in range(1, 4):
            try:
                return func(*args, **kwargs)
            except (requests.ConnectionError, requests.Timeout) as e:
                print(f"[retry] {attempt}번째 실패: {type(e).__name__}")
                if attempt == 3:
                    raise
                time.sleep(1)
        return wrapper

파이썬에서 **if**문과 **raise**를 함께 쓰면 특정 조건일 때 의도적으로 예외(에러)를 발생시킬 수 있습니다.

In [29]:
def set_age(age):
    if age < 0:
        raise ValueError("나이는 0 이상")
    return age

set_age(-1)

ValueError: 나이는 0 이상

In [72]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import sys
import urllib.request

import json

client_id = "5ZVSY6k06TO2UNTdua0g"
client_secret = "qDGN8j9CQl"
encText = urllib.parse.quote("둠스데이")
url = "https://openapi.naver.com/v1/search/blog?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    #print(response_body.decode('utf-8'))
    res_body = response_body.decode('utf-8')
    print(type(response_body))
    print("read 완료\n")
else:
    print("Error Code:" + rescode)


print(type(res_body))
data_json = json.loads(res_body)
#data_json["items"]
result = [i["bloggername"] for i in data_json["items"]] 

<class 'bytes'>
read 완료

<class 'str'>


In [60]:
data_json["items"][0]["bloggername"]

'엔터로그 (ENTER LOG)'

In [69]:
print(type(result))
print(*result, sep="\n")

<class 'list'>
엔터로그 (ENTER LOG)
돈버릇: 부자가 되는 건강한 소비 습관
오늘도 엔터톡
잔잔한 기록❤️
트렌드한입
도라마|드라마.스타이슈
하루정보노트
오늘 이슈 노트
Movie:graphy
콘텐츠레이더


In [71]:
data_json

{'lastBuildDate': 'Fri, 27 Nov 2026 15:22:15 +0900',
 'total': 32237,
 'start': 1,
 'display': 10,
 'items': [{'title': '어벤져스 <b>둠스데이</b> 개봉일·예매 전 꼭 알아야 할 등장인물 총정리 (....',
   'link': 'https://blog.naver.com/wjdgh0903/224487920168',
   'description': '결론부터 말씀드리면 어벤져스 <b>둠스데이</b>, 이제 진짜 코앞이에요.... 개봉일부터 예매 전략까지, 날짜가 생명이다 어벤져스 <b>둠스데이</b>는... 어벤져스 <b>둠스데이</b>가 그의 진짜 마지막 작품이 됐거든요. 엑스맨... ',
   'bloggername': '엔터로그 (ENTER LOG)',
   'bloggerlink': 'https://blog.naver.com/wjdgh0903',
   'postdate': '20261123'},
  {'title': '어벤져스 <b>둠스데이</b> 개봉일 예매 일정과 러닝타임 등장인물 총정리',
   'link': 'https://blog.naver.com/loversmin/224490590911',
   'description': '어벤져스 <b>둠스데이</b> 메인 예고편이 공개되면서 극장가 열기가... 극장 관람을 계획하고 계신 분들을 위해 어벤져스 <b>둠스데이</b> 개봉일과... 어벤져스 <b>둠스데이</b> 기본 정보 및 예매 일정 요약 어벤져스 <b>둠스데이</b>... ',
   'bloggername': '돈버릇: 부자가 되는 건강한 소비 습관',
   'bloggerlink': 'https://blog.naver.com/loversmin',
   'postdate': '20261124'},
  {'title': '어벤져스 <b>둠스데이</b> 개봉일 12월 16일 확정, 닥터 둠·출연진·예고편 ....',
   'link': 'https: